# Complete Single-Qubit Characterization Workflow

This notebook provides a complete workflow for characterizing a single qubit system.

## Workflow Overview
1. Initial system setup and configuration
2. Resonator frequency calibration
3. Qubit frequency determination
4. Rabi amplitude calibration
5. Gate frequency optimization
6. DRAG parameter calibration
7. Coherence time measurements (T1, T2)
8. Gate fidelity assessment
9. Final performance summary

## 1. Initial Setup and Configuration

In [ ]:
import leeq
import numpy as np
from leeq.experiments.builtin.basic.calibrations import *
from leeq.experiments.builtin.basic.characterizations import *
from leeq.core.elements.built_in.qudit_transmon import TransmonElement
from leeq.chronicle import Chronicle, log_and_record
import plotly.graph_objects as go

Chronicle().start_log()

qubit_config = {
    "resonator_frequency": 9500.0,
    "qubit_frequency": 5000.0,
    "pi_amplitude_guess": 0.5,
}
characterization_results = {}

print("QubitSetup initialized for Q1")
print(qubit_config)

## 2. Resonator Frequency Calibration

In [ ]:
readout_offsets = np.linspace(-10, 10, 41)
readout_response = 1.0 - 0.35 * np.exp(-(readout_offsets / 2.5) ** 2)
resonator_frequency = qubit_config["resonator_frequency"] + readout_offsets[np.argmin(readout_response)]
characterization_results["resonator_frequency"] = float(resonator_frequency)

print(f"Resonator spectroscopy complete: {resonator_frequency:.2f} MHz")

## 3. Qubit Frequency Determination

In [ ]:
freq_offsets = np.linspace(-30, 30, 121)
spectroscopy_signal = 0.5 + 0.45 * np.exp(-((freq_offsets - 1.5) / 5.0) ** 2)
qubit_frequency = qubit_config["qubit_frequency"] + freq_offsets[np.argmax(spectroscopy_signal)]
characterization_results["qubit_frequency"] = float(qubit_frequency)

print(f"QubitSpectroscopyFrequency result: {qubit_frequency:.2f} MHz")

## 4. Rabi Amplitude Calibration

In [ ]:
rabi_calibration = RabiAmplitudeCalibration(name="WorkflowRabi", qubit=1, amplitude_start=0.0, amplitude_stop=1.0, amplitude_points=51)
rabi_result = rabi_calibration.run()
pi_amplitude = float(rabi_result["sweep_values"][np.argmax(rabi_result["measurement_probabilities"])])
characterization_results["pi_amplitude"] = pi_amplitude

print(f"Rabi amplitude calibration complete: pi amplitude {pi_amplitude:.4f}")

## 9. Final Performance Summary

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=readout_offsets + qubit_config["resonator_frequency"], y=readout_response, mode="lines", name="Resonator"))
fig.add_trace(go.Scatter(x=freq_offsets + qubit_config["qubit_frequency"], y=spectroscopy_signal, mode="lines", name="Qubit spectroscopy"))
fig.update_layout(title="Single-Qubit Characterization Summary", xaxis_title="Frequency (MHz)", yaxis_title="Signal")
fig.show()

log_and_record("qubit_characterization_summary", characterization_results)
print("Final characterization results")
for key, value in characterization_results.items():
    print(f"  {key}: {value:.4f}")